# Modelado Predictivo: Vector Error Correction Model (VECM)

## 1. Importación de Librerías y Configuración del Entorno
En esta primera etapa procedemos a importar las librerías necesarias para el análisis y modelado. Para la manipulación y visualización de los datos utilizaremos `Pandas`, `NumPy`, `Matplotlib` y `Seaborn`. 

Dado el rigor estadístico requerido para el modelo VECM, nos apoyaremos fuertemente en el módulo de series temporales de la librería `Statsmodels`. De ella extraeremos las pruebas de raíces unitarias (Test de Dickey-Fuller), las pruebas de cointegración (Test de Johansen) y la arquitectura base del algoritmo VECM.

In [1]:
# ==============================================================================
# Celda 1: Importación de librerías para el modelo VECM
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías estadísticas de Statsmodels
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM, select_order
from statsmodels.tools.eval_measures import rmse, mse

import warnings
warnings.filterwarnings("ignore")

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Carga de Datos y Estructuración de la Serie Temporal
Para el correcto funcionamiento de los modelos de `Statsmodels`, es imperativo que el conjunto de datos posea un índice de tipo `datetime` continuo y con una frecuencia explícita. En este caso, forzaremos la frecuencia a formato horario (`'h'`).

Asimismo, el modelo VECM tradicional está diseñado para analizar las relaciones a largo plazo entre variables **endógenas** (aquellas que se influyen mutuamente y que queremos predecir, como la intensidad, la velocidad y la ocupación). Por ello, en esta fase separaremos las variables endógenas de las características temporales de calendario (hora, día de la semana, etc.), ya que estas últimas introducirían ruido determinista en el test de cointegración.

In [ ]:
# ==============================================================================
# Celda 2: Carga de particiones y configuración del índice temporal
# ==============================================================================
# Rutas a los datos (ajusta si la ruta relativa en tu PC es distinta)
path_train = '../../data/processed/Split_Datasets/data_train.csv'
path_val   = '../../data/processed/Split_Datasets/data_val.csv'
path_test  = '../../data/processed/Split_Datasets/data_test.csv'

def load_and_prep(path):
    df = pd.read_csv(path)
    # Convertir 'fecha' a tipo datetime y establecerla como índice
    df['fecha'] = pd.to_datetime(df['fecha'])
    df.set_index('fecha', inplace=True)
    
    # Asegurar que la frecuencia es horaria (importante para series temporales)
    df = df.asfreq('h')
    return df

train_df = load_and_prep(path_train)
val_df   = load_and_prep(path_val)
test_df  = load_and_prep(path_test)

# Separar variables endógenas (intensidad, ocupacion, vmed) de las de calendario
cols_endogenas = [col for col in train_df.columns if col not in ['hora', 'dia_semana', 'mes', 'es_finde']]
train_endo = train_df[cols_endogenas]

print(f"Dimensiones de Entrenamiento (Endógenas): {train_endo.shape}")
print("Variables a analizar:", list(train_endo.columns))
train_endo.head(3)